# 00 Data Loading And Split

            Use this notebook first.
            It is the place to:

            - inspect exported `episode_frames`
            - build the train-ready sample table
            - save one shared train/val/test split file
            - reuse that same split for every model notebook


In [5]:
import json
from pathlib import Path
import sys

PROJECT_ROOT = Path.home() / "Documents/Thesis"
PIPELINE_ROOT = PROJECT_ROOT / "model_training"
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from datasets.paths import EPISODE_FRAMES_ROOT, SPLITS_ROOT, TRAIN_READY_ROOT
from datasets.sample_table import build_sample_table, load_episode_streams, save_or_load_fixed_split, save_sample_table
from training.notebook_workflow import describe_shared_artifacts, finish_runtime_report, start_runtime_report

SEED = 42
PAST_LEN = 10
FUTURE_LEN = 5
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

print("Pipeline root:", PIPELINE_ROOT)
print("Episode frames root:", EPISODE_FRAMES_ROOT)
print("Split strategy: episode-level shared split")



Pipeline root: /home/basudeo/Documents/Thesis/model_training
Episode frames root: /home/basudeo/Documents/Thesis/model_training/results/episode_frames
Split strategy: episode-level shared split


In [6]:
sample_files = sorted(EPISODE_FRAMES_ROOT.glob("*/frames.jsonl"))
print("Episode files:", len(sample_files))
for path in sample_files[:10]:
    print(path)


Episode files: 8
/home/basudeo/Documents/Thesis/model_training/results/episode_frames/run_model_20260522_201207/frames.jsonl
/home/basudeo/Documents/Thesis/model_training/results/episode_frames/run_model_20260522_201854/frames.jsonl
/home/basudeo/Documents/Thesis/model_training/results/episode_frames/run_model_20260522_202537/frames.jsonl
/home/basudeo/Documents/Thesis/model_training/results/episode_frames/run_model_20260522_203433/frames.jsonl
/home/basudeo/Documents/Thesis/model_training/results/episode_frames/run_model_20260522_204215/frames.jsonl
/home/basudeo/Documents/Thesis/model_training/results/episode_frames/run_model_20260523_013116/frames.jsonl
/home/basudeo/Documents/Thesis/model_training/results/episode_frames/run_model_20260523_013946/frames.jsonl
/home/basudeo/Documents/Thesis/model_training/results/episode_frames/run_model_20260523_015017/frames.jsonl


In [7]:
data_runtime = start_runtime_report(
    stage_name="data_loading_and_split",
    output_dir=PIPELINE_ROOT / "results",
    context={
        "seed": SEED,
        "past_len": PAST_LEN,
        "future_len": FUTURE_LEN,
        "train_ratio": TRAIN_RATIO,
        "val_ratio": VAL_RATIO,
    },
)
streams = load_episode_streams(EPISODE_FRAMES_ROOT)
sample_table = build_sample_table(streams, past_len=PAST_LEN, future_len=FUTURE_LEN)

sample_table_path = TRAIN_READY_ROOT / f"sample_table_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json"
split_path = SPLITS_ROOT / f"trajectory_split_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json"

save_sample_table(sample_table, sample_table_path)
split_info = save_or_load_fixed_split(
    sample_table=sample_table,
    split_path=split_path,
    seed=SEED,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    past_len=PAST_LEN,
    future_len=FUTURE_LEN,
)

summary = {
    "stream_count": len(streams),
    "sample_count": len(sample_table),
    "sample_table_path": str(sample_table_path),
    "split_path": str(split_path),
    "split_strategy": split_info["split_strategy"],
    "episode_count": split_info["episode_count"],
    "train_episode_ids": split_info["train_episode_ids"],
    "val_episode_ids": split_info["val_episode_ids"],
    "test_episode_ids": split_info["test_episode_ids"],
    "train_count": len(split_info["train_indices"]),
    "val_count": len(split_info["val_indices"]),
    "test_count": len(split_info["test_indices"]),
}
data_runtime_summary = finish_runtime_report(
    data_runtime,
    extra={
        **summary,
        **describe_shared_artifacts(
            streams=streams,
            sample_table=sample_table,
            split_info=split_info,
            sample_table_path=sample_table_path,
            split_path=split_path,
        ),
    },
)
print(json.dumps(summary, indent=2))
print("\nRuntime summary:")
print(json.dumps(data_runtime_summary, indent=2))



[runtime] data_loading_and_split start: 2026-05-23T11:22:17 summary=/home/basudeo/Documents/Thesis/model_training/results/runtime/20260523_112217_data_loading_and_split_runtime_summary.json
[runtime] data_loading_and_split done: start=2026-05-23T11:22:17 end=2026-05-23T11:22:21 elapsed_s=4.203 peak_cpu=16.30% peak_rss_mb=1279.49 peak_gpu_alloc_mb=0.00
{
  "stream_count": 8,
  "sample_count": 51706,
  "sample_table_path": "/home/basudeo/Documents/Thesis/model_training/results/train_ready/sample_table_seed42_past10_future5.json",
  "split_path": "/home/basudeo/Documents/Thesis/model_training/results/splits/trajectory_split_seed42_past10_future5.json",
  "split_strategy": "episode",
  "episode_count": 8,
  "train_episode_ids": [
    "run_model_20260522_203433",
    "run_model_20260522_204215",
    "run_model_20260523_013946",
    "run_model_20260523_015017",
    "run_model_20260522_202537"
  ],
  "val_episode_ids": [
    "run_model_20260523_013116"
  ],
  "test_episode_ids": [
    "run_mo

In [8]:
print("First sample:")
print(json.dumps(sample_table[0], indent=2)[:2000])

print("\nShared split preview:")
preview = {
    "train_sample_id": [split_info["sample_ids"][idx] for idx in split_info["train_indices"][:5]],
    "val_sample_id": [split_info["sample_ids"][idx] for idx in split_info["val_indices"][:5]],
    "test_sample_id": [split_info["sample_ids"][idx] for idx in split_info["test_indices"][:5]],
}
print(json.dumps(preview, indent=2))


First sample:
{
  "sample_id": "run_model_20260522_201207_stream000_start00000",
  "episode_id": "run_model_20260522_201207",
  "stream_index": 0,
  "start_index": 0,
  "anchor_index": 9,
  "past_len": 10,
  "future_len": 5,
  "anchor_timestamp_ns": 1779473541508434400,
  "teacher_state": "reverse",
  "future_xy_local": [
    [
      -0.0012023469643292694,
      1.3336727970492232e-05
    ],
    [
      -0.001731696631942069,
      1.9198863708294882e-05
    ],
    [
      -0.0024011580426795886,
      2.6604224847052277e-05
    ],
    [
      -0.002893653682911118,
      3.2046029976504986e-05
    ],
    [
      -0.004086710669118454,
      4.5207355754896044e-05
    ]
  ],
  "future_dt": [
    0.077933737,
    0.193541016,
    0.26439991500000004,
    0.302446074,
    0.373393668
  ]
}

Shared split preview:
{
  "train_sample_id": [
    "run_model_20260522_202537_stream002_start00000",
    "run_model_20260522_202537_stream002_start00001",
    "run_model_20260522_202537_stream002_sta